In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# from google.colab.patches import cv2_imshow
# import cv2
# import numpy as np

# # Загрузка изображения
# img = cv2.imread("/content/road.png")
# hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# # Маска серого (асфальт)
# lower_gray = np.array([0, 0, 40])
# upper_gray = np.array([180, 50, 130])
# mask = cv2.inRange(hsv, lower_gray, upper_gray)

# # Выделенная область дороги
# result = cv2.bitwise_and(img, img, mask=mask)

# # Показ результатов
# cv2_imshow(img)       # оригинал
# cv2_imshow(mask)      # маска


error: OpenCV(4.11.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from tqdm import tqdm
!apt-get install unrar     # для .rar
!apt-get install unzip     # для .zip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
#!unrar x "/content/drive/MyDrive/big_dataset_for_sdc/MergedData.rar" "/content/drive/MyDrive/clear_data_set"

###Параметры и классы

In [ ]:

DATA_DIR = "/content/drive/MyDrive/clear_data_set/output_images"
LABELS_CSV = "/content/drive/MyDrive/clear_data_set/labels.csv"
SAVE_DIR = "/content/drive/MyDrive/models_for_sdc"
os.makedirs(SAVE_DIR, exist_ok=True)
NUM_CLASSES = 8
BATCH_SIZE = 1
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rare_classes = ["RezkiyLeft", "RezkiyRight", "stop","slow"]

class_names = {
    0: "straight",
    1: "left",
    2: "right",
    3: "RezkiyLeft",
    4: "RezkiyRight",
    5: "stop",
    6: "slow",
    7: "speed_up"
}

label_to_idx = {v: k for k, v in class_names.items()}

###Dataset

In [ ]:
class DrivingDataset(Dataset):
    def __init__(self, df, images_dir, transform_common=None, transform_rare=None, rare_classes=None):
        """
        df: DataFrame с колонками ['filename', 'action']
        images_dir: путь к папке, где лежат изображения
        transform: torchvision.transforms для обработки изображения
        """
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform_common = transform_common
        self.transform_rare = transform_rare
        self.rare_classes = set(rare_classes or [])


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["filename"])
        # Открываем PIL-объект, чтобы сохранить цветность
        image = Image.open(img_path).convert("RGB")
        label_str = row["label"]
        label = label_to_idx[label_str]

        if label_str in self.rare_classes and self.transform_rare:
            image = self.transform_rare(image)
        elif self.transform_common:
            image = self.transform_common(image)


        return image, label


### Masked DataSet


In [ ]:
class DrivingDataset(Dataset):
    def __init__(self, df, images_dir, transform_common=None, transform_rare=None, rare_classes=None):
        """
        df: DataFrame с колонками ['filename', 'action']
        images_dir: путь к папке, где лежат изображения
        transform: torchvision.transforms для обработки изображения
        """
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.transform_common = transform_common
        self.transform_rare = transform_rare
        self.rare_classes = set(rare_classes or [])


    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["filename"])

        # Открываем PIL-объект, чтобы сохранить цветность
        image = Image.open(img_path).convert("RGB")
        label_str = row["action"]
        label = label_to_idx[label_str]

        # --- 1. Чтение и выделение дороги через маску ---
        image_bgr = cv2.imread(img_path)
        hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

        # Маска для асфальта и бордюров
        lower_gray = np.array([0, 0, 40])
        upper_gray = np.array([180, 50, 130])
        mask = cv2.inRange(hsv, lower_gray, upper_gray)

        # Переводим в RGB → PIL.Image
        road_only_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        road_image_pil = Image.fromarray(road_only_rgb)


        if label_str in self.rare_classes and self.transform_rare:
            image = self.transform_rare(image)
        elif self.transform_common:
            image = self.transform_common(image)


        return image, label


###Transformations

In [ ]:
transform_common = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# transform_rare = T.Compose([
#     T.Resize((224, 224)),
#     T.RandomHorizontalFlip(),
#     T.RandomRotation(15),
#     T.ColorJitter(
#     brightness=0.2,
#     contrast=0.2,
#     saturation=0.2,
#     hue=0.1
# ),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]),
# ])


In [ ]:
!ls /content/drive/MyDrive/data_for_sdc/


###Чтение CSV и разбиение

In [ ]:
df_all = pd.read_csv(LABELS_CSV, sep=",")  # ожидается столбцы: filename,label

# Проверка на наличие файлов
missing_files = [f for f in df_all["filename"] if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing_files:
    print("Внимание! Отсутствуют файлы:", missing_files[:5])
    # Можно либо падать, либо фильтровать
    df_all = df_all[df_all["filename"].isin(set(df_all["filename"]) - set(missing_files))]

In [ ]:
# Train/Val split (80/20)
df_train, df_val = train_test_split(df_all, test_size=0.2, random_state=42, stratify=df_all["label"])

train_dataset = DrivingDataset(df_train, DATA_DIR, transform_common=transform_common, transform_rare=transform_common, rare_classes=rare_classes)
val_dataset = DrivingDataset(df_val, DATA_DIR, transform_common=transform_common, transform_rare=transform_common, rare_classes=rare_classes)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")


Train size: 3758, Val size: 940


 ### Модель (MobileNetV2 с 8 выходами)

In [ ]:
# Загружаем MobileNetV2 без весов (либо с предобученными, если хотите)
weights = MobileNet_V2_Weights.DEFAULT
model = mobilenet_v2(weights=weights)
model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
model = model.to(DEVICE)

# Оптимизатор и loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 23.6MB/s]


### Train stage


In [ ]:
def train_one_epoch(loader, model, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)              # shape [B, 8]
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [ ]:
def validate(loader, model, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [ ]:
import matplotlib.pyplot as plt

best_val_acc = 0.0

# ⬇️ Логгируем метрики для графиков
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc="training"):
    train_loss, train_acc = train_one_epoch(train_loader, model, criterion, optimizer, DEVICE)
    val_loss, val_acc = validate(val_loader, model, criterion, DEVICE)

    # 🔢 Добавляем в историю
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
          f"Val loss: {val_loss:.4f}, acc: {val_acc:.4f}")

    # 💾 Сохраняем модель
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_path = os.path.join(SAVE_DIR, "mobilenet8_best.pth")
        torch.save(model.state_dict(), save_path)
        print(f"→ Сохранена лучшая модель (val_acc={best_val_acc:.4f}) в {save_path}")

print("✅ Обучение завершено.")

# 📈 Строим графики после обучения
plt.figure(figsize=(12, 5))

# График лосса
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over epochs")
plt.legend()
plt.grid(True)

# График accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over epochs")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

training:   0%|          | 0/10 [00:00<?, ?it/s]